In [1]:
import copy
import datetime
import numpy as np
import os
import pandas as pd
import polars as pl
import requests
import zipfile

from tqdm import tqdm
from typing import Tuple, Dict, List, Any

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [2]:
'''
Основные компоненты SASRec:
1. Embedding Layer: каждому объекту сопоставляется обучаемый эмбеддинг фиксированной размерности d и добавляется позиционный эмбеддинг для кодирования позиции объекта.
2. Transformer Encoder: последовательное применение l блоков энкодера трансформера с маской.
   Каждый блок содержит: слой self-attention, слой position-wise feed-forward, также: residual connections и layernorm.
3. Prediction Layer: для каждой позиции t выход энкодера скалярно умножается на эмбеддинг каждого объекта, чтобы получить скор релевантности.
'''

'\nОсновные компоненты SASRec:\n1. Embedding Layer: каждому объекту сопоставляется обучаемый эмбеддинг фиксированной размерности d и добавляется позиционный эмбеддинг для кодирования позиции объекта.\n2. Transformer Encoder: последовательное применение l блоков энкодера трансформера с маской. \n   Каждый блок содержит: слой self-attention, слой position-wise feed-forward, также: residual connections и layernorm. \n3. Prediction Layer: для каждой позиции t выход энкодера скалярно умножается на эмбеддинг каждого объекта, чтобы получить скор релевантности. \n'

In [52]:
def download(url: str, filename: str, chunk_size: int = 1024):
    response = requests.get(url, stream=True)
    response.raise_for_status()

    with open(filename, "wb") as f:
        for chunk in response.iter_content(chunk_size=chunk_size):
            if chunk:
                f.write(chunk)

    with zipfile.ZipFile(filename, "r") as zip_ref:
        zip_ref.extractall(".")

In [53]:
download(
    url="http://files.grouplens.org/datasets/movielens/ml-1m.zip",
    filename="ml-1m.zip"
)

In [54]:
dataset_path = './ml-1m/'

ratings = pd.read_csv(
    os.path.join(dataset_path, 'ratings.dat'),
    delimiter='::',
    header=None,
    names=['user_id', 'movie_id', 'rating', 'timestamp'],
    engine='python'
)
ratings = pl.from_pandas(ratings)

movies = pd.read_csv(
    os.path.join(dataset_path, 'movies.dat'),
    delimiter='::',
    header=None,
    names=['movie_id', 'title', 'genres'],
    encoding='latin-1',
    engine='python'
)
movies = pl.from_pandas(movies)

users = pd.read_csv(
    os.path.join(dataset_path, 'users.dat'),
    delimiter='::',
    header=None,
    names=['user_id', 'gender', 'age', 'occupation', 'zip_code'],
    engine='python'
)
users = pl.from_pandas(users)

In [55]:
filtering_stage = 0
is_changed = True
threshold = 5
good_users = set()
good_items = set()

filtered_ratings = ratings.clone()

# фильтруем данные
# удаляем пользователей и фильмы, у которых количество взаимодейсвий меньше определенного порога
while is_changed:
    user_counts = filtered_ratings.group_by('user_id').agg(pl.len().alias('user_count'))
    movie_counts = filtered_ratings.group_by('movie_id').agg(pl.len().alias('movie_count'))

    good_users = user_counts.filter(pl.col('user_count') >= threshold).select('user_id')
    good_movies = movie_counts.filter(pl.col('movie_count') >= threshold).select('movie_id')

    old_size = len(filtered_ratings)

    # после фильтрации соединяем две таблицы с помощью inner join
    new_ratings = filtered_ratings.join(good_users, on='user_id', how='inner')
    new_ratings = new_ratings.join(good_movies, on='movie_id', how='inner')

    new_size = len(new_ratings)

    filtered_ratings = new_ratings
    is_changed = old_size != new_size
    filtering_stage += 1

filtered_ratings = filtered_ratings.with_columns(movie_id = pl.col('movie_id').rank('dense') - 1)
filtered_ratings = filtered_ratings.sort(['user_id', 'timestamp'])

grouped_filtered_df = filtered_ratings.group_by('user_id', maintain_order=True).agg(
    pl.all().exclude('user_id')
)

In [56]:
# разделение выборки
valid_portion = 0.1
test_portion = 0.1

all_events_timestamp = []
for _, _, _, timestamp in filtered_ratings.iter_rows():
    all_events_timestamp.append(timestamp)

all_events_timestamp = sorted(all_events_timestamp)

fst_threshold = all_events_timestamp[int(len(all_events_timestamp) * (1.0 - test_portion - valid_portion))]
snd_threshold = all_events_timestamp[int(len(all_events_timestamp) * (1.0 - test_portion))]


In [57]:
train_data = []
valid_data = []
eval_data = []

# разделение данных по времени
for user_id, item_history, rating, timestamp in grouped_filtered_df.iter_rows():
    train_history = []
    history = []
    history_ts = []

    # итерация по парам: (фильм, время)
    for item_id, ts in zip(item_history, timestamp):
        # этап обучения
        if ts < fst_threshold:
            assert len(history) == 0 or ts >= history_ts[-1]
            train_history.append(item_id)
        # этап валидации
        elif ts < snd_threshold:
            assert len(history) == 0 or ts >= history_ts[-1]
            if len(history) >= 1:
                valid_data.append({
                    'user_id': user_id,
                    'history': [x for x in history + [item_id]]
                })
        else:
            assert len(history) == 0 or ts >= history_ts[-1]
            if len(history) >= 1:
                eval_data.append({
                    'user_id': user_id,
                    'history': [x for x in history + [item_id]]
                })

        history.append(item_id)
        history_ts.append(ts)

    if len(train_history) >= 5:
        train_data.append({
            'user_id': user_id,
            'history': train_history
        })

In [58]:
# Реализация Dataset для поставки данных в модель:
class MovieLensDataset(Dataset):
    def __init__(self, data: List[Dict], num_items: int, max_seq_len: int) -> None:
        super().__init__()

        # data - список словарей. Каждый словарь содержит элемент с ключом 'history' - это список идентификаторов фильмов.
        self.data = data
        self.num_items = num_items # общее количество уникальных фильмов
        self.max_seq_len = max_seq_len # максимальная длина последовательности, которую может использовать модель.

    def __len__(self) -> int: # возвращает количество пользователей в self.data
        return len(self.data)

    def __getitem__(self, index: int) -> Dict[str, List[int]]:
        # принимает index и извлекает соответствующую историю
        sample = self.data[index]
        items = list(map(int, sample['history']))

        result = {}


        # формируется входная последовательность: берется вся история без последнего взаимодействия, чтобы предсказывать его и подается в модель.
        item_sequence = items[:-1][-self.max_seq_len:]

        # формируется ответ, который нужно предказать для входной последовательности: это входная последовательность, сдвинутая на 1.
        next_item_sequence = items[1:][-self.max_seq_len:]

        # генерируется случайный отрицательный пример
        random_negative_ids = np.random.randint(0, self.num_items, size=(len(item_sequence,))).tolist()

        result['history'] = dict(
            item_ids=item_sequence, # входная последовательность
            lengths=len(item_sequence), # длина последовательности
            positions=np.arange(start=len(item_sequence) - 1, stop=-1, step=-1).tolist() # позиции элементов
        )

        # последовательность, которая должна получиться на выходе
        result['positives'] = dict(item_ids=next_item_sequence)

        # отрицательный пример
        result['negatives'] = dict(item_ids=random_negative_ids)

        return result

In [59]:
'''
Объединение списка словарей в один батч для поставки в модель.
'''
def collate_fn(batch: List[Dict[str, Any]]) -> Dict[str, Any]:
    result = {}
    keys = batch[0].keys()
    for key in keys:
        values = [item[key] for item in batch]
        first = values[0]
        if isinstance(first, dict):
            # если значение словарь - рекурсивно применить функцию
            result[key] = collate_fn(values)
        elif isinstance(first, list):
            # если значение список - объединение в тензор
            result[key] = torch.tensor(sum(values, []), dtype=torch.long)
        else:
            # создание тензора в остальных случаях
            result[key] = torch.tensor(values, dtype=torch.long)
    return result

In [60]:
'''
Создание объекта Dataset.
Передаём список словарей, которые содержат историю взаимодействия пользователей.
NUM_ITEMS - количество фильмов для генерации уникальных значений.
MAX_SEQ_LEN - максимальная длина последовательности для подачи в модель.
'''

NUM_ITEMS = 3416
MAX_SEQ_LEN = 5

tmp_dataset = MovieLensDataset(data=train_data, num_items=NUM_ITEMS, max_seq_len=MAX_SEQ_LEN)
tmp_iter = iter(tmp_dataset) # создание итератора

In [61]:
'''
Создание DataLoader для загрузки в модель бачами по 3 элемента.
'''
BATCH_SIZE = 3

tmp_dataloader = DataLoader(dataset=tmp_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=False, collate_fn=collate_fn)

In [62]:
# создание маски

def get_mask(lengths: torch.Tensor) -> torch.Tensor:
    maxlen = lengths.max().item()
    return torch.arange(maxlen, device=lengths.device).expand(len(lengths), maxlen) < lengths.unsqueeze(1)


def get_last(data: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
    offsets = torch.cumsum(lengths, dim=-1)
    return data[offsets - 1]

In [63]:
# преобразование истории взаимодействия пользователя в эмбеддинги
class SasRecBackbone(nn.Module):
    def __init__(self, num_items: int, embedding_dim: int = 64, num_heads: int = 2, max_seq_len: int = 512, dropout_rate: float = 0.2, num_transformer_layers: int = 2) -> None:
        super().__init__()
        self.num_items = num_items
        self.num_heads = num_heads
        self.embedding_dim = embedding_dim
        self.item_encoder = nn.Embedding(
            num_embeddings=num_items,
            embedding_dim=embedding_dim
        )
        self.position_embeddings = nn.Embedding(num_embeddings=max_seq_len, embedding_dim=embedding_dim) # эмбеддинги, кодирующие позиции элементов


        # несколько слоёв трансформера
        encoder_layer = nn.TransformerEncoderLayer(d_model=embedding_dim, nhead=num_heads,dim_feedforward=embedding_dim * 4, dropout=dropout_rate, activation='gelu', batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_transformer_layers)

    # получение весов
    @property
    def catalog_embeddings(self) -> torch.Tensor:
        return self.item_encoder.weight.data

    # inputs - входная последовательность
    def forward(self, inputs: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
        item_embeddings = self.item_encoder(inputs['history']['item_ids'])  # получение эмбеддингов
        padding_mask = get_mask(inputs['history']['lengths'])  # создание маски
        batch_size, seq_len = padding_mask.shape

        # создание тензора токенов с позиционными эмбеддингами
        token_embeddings = item_embeddings.new_zeros(batch_size, seq_len, self.embedding_dim)
        token_embeddings[padding_mask] = (item_embeddings + self.position_embeddings(inputs['history']['positions']))

        # применение трансформера
        encoder_output = self.transformer_encoder(
            src=token_embeddings,
            mask=torch.triu(
                torch.ones((seq_len, seq_len), dtype=torch.bool, device=token_embeddings.device),
                diagonal=1
            ),
            src_key_padding_mask=~padding_mask,
            is_causal=True
        )[padding_mask]

        return {
            'encoder_output': encoder_output,
        }

In [64]:
class SASRecModel(nn.Module):
    def __init__(self, backbone: SasRecBackbone) -> None:
        super().__init__()
        self.backbone = backbone
        self.init_weights(0.02)

    # начальная инициализация весов
    @torch.no_grad()
    def init_weights(self, initializer_range: float = 0.02) -> None:

        # проход по всем парамерам модели
        for key, value in self.named_parameters():
            if 'weight' in key:
                if 'norm' in key:
                    nn.init.ones_(value.data) # устанавливаем значения в 1, если есть нормализация
                else:
                    nn.init.trunc_normal_(value.data, std=initializer_range, a=-2 * initializer_range, b=2 * initializer_range) # инициализируем значениями из нормального распределения
            elif 'bias' in key:
                nn.init.zeros_(value.data)

    def forward(self, inputs: Dict):
        backbone_outputs = self.backbone(inputs) # входные данные передаются в SasRecBackbone для получения эмбеддингов. Будет возвращен словарь с ключом: 'encoder_output', содержащий эмбеддинги

        if self.training:
            # процесс обучения. Вычисление лосса.
            loss = self.compute_loss(inputs, backbone_outputs)
            return {
                'loss': loss
            }
        else:
            # режим валидации. вычисление скоров релевантности.
            last_embeddings = get_last(
                data=backbone_outputs['encoder_output'],
                lengths=inputs['history']['lengths']
            )
            candidate_scores = last_embeddings @ self.backbone.catalog_embeddings.T
            return {
                'candidate_scores': candidate_scores
            }

In [65]:
class SasRecReal(SASRecModel):

    # вычисление функции потерь
    def compute_loss(self, inputs: Dict, backbone_output: Dict) -> torch.Tensor:
        query_embeddings = backbone_output['encoder_output'] # получение эмбеддингов
        positive_embeddings = self.backbone.item_encoder(inputs['positives']['item_ids'])
        negative_embeddings = self.backbone.item_encoder(inputs['negatives']['item_ids']) # эмбеддинги случайно выбранных отрицательных элементов

        # вычисление скалярных произведений
        positive_scores = (query_embeddings * positive_embeddings).sum(dim=-1)
        negative_scores = (query_embeddings * negative_embeddings).sum(dim=-1)

        # вычисление бинарной кросс-энтропии
        loss = torch.nn.functional.binary_cross_entropy_with_logits(
            input=torch.cat([positive_scores, negative_scores], dim=0),
            target=torch.cat([torch.ones_like(positive_scores), torch.zeros_like(negative_scores)]),
            reduction='mean'
        )

        return loss

In [66]:
# функции подсчета метрик

# вычисление метрики Hit Rate
def compute_hitrate(indices: torch.Tensor, labels: torch.Tensor, k: int) -> List[float]:
    assert indices.shape[0] == labels.shape[0]
    hits = torch.eq(indices[:, :k], labels[..., None]).float() # берем первые k предсказаний
    hitrate = hits.sum(dim=-1) # суммирование по последнему измерению
    return np.mean(hitrate.cpu().tolist()) # вычисление среднего

# вычисление DCG
def compute_dcg(indices: torch.Tensor, labels: torch.Tensor, k: int) -> List[float]:
    assert indices.shape[0] == labels.shape[0]
    hits = torch.eq(indices[:, :k], labels[..., None]).float() # берем первые k предсказаний
    discount_factor = 1 / torch.log2(torch.arange(1, k + 1, 1).float() + 1.).to(hits.device)
    dcg = torch.einsum('bk,k->b', hits, discount_factor) # умножение каждой строки матрицы hits на вектор discount_factor
    return np.mean(dcg.cpu().tolist())


def compute_coverage(indices: torch.Tensor, k: int, num_items: int = NUM_ITEMS) -> List[float]:
    return indices[:, :k].reshape(-1).unique().shape[0] / num_items


def compute_metrics(indices: torch.Tensor, labels: torch.Tensor) -> Dict[str, float]:
    return {
        'dcg@5': compute_dcg(indices, labels, k=5),
        'dcg@10': compute_dcg(indices, labels, k=10),
        'dcg@20': compute_dcg(indices, labels, k=20),

        'hitrate@5': compute_hitrate(indices, labels, k=5),
        'hitrate@10': compute_hitrate(indices, labels, k=10),
        'hitrate@20': compute_hitrate(indices, labels, k=20),

        'coverage@5': compute_coverage(indices, k=5),
        'coverage@10': compute_coverage(indices, k=10),
        'coverage@20': compute_coverage(indices, k=20),
    }

In [67]:
# получение предсказаний на валидации
def evaluation(dataloader: DataLoader, model: SASRecModel, device: str = 'cpu') -> Tuple[torch.Tensor, torch.Tensor]:
    outputs = []
    labels = []

    # проход по каждому батчу DataLoader
    for batch in dataloader:
        model.eval()

        for key in batch:
            if isinstance(batch[key], dict):
                for sub_key in batch[key]:
                    batch[key][sub_key] = batch[key][sub_key].to(device)

            else:
                assert isinstance(batch[key], torch.Tensor)
                batch[key] = batch[key].to(device)


        # передаём модели батч
        candidate_scores = model(batch)['candidate_scores']

        # вычисление топ 20 кандидатов
        _, top_candiate_ids = torch.topk(
            candidate_scores,
            k=20, dim=-1, largest=True
        )

        next_item_ids = get_last(
            data=batch['positives']['item_ids'],
            lengths=batch['history']['lengths']
        )
        outputs.append(top_candiate_ids)
        labels.append(next_item_ids)

    return torch.cat(outputs, dim=0), torch.cat(labels, dim=0)

In [68]:
def train(train_dataloader: DataLoader, valid_dataloader: DataLoader, model: torch.nn.Module, optimizer: torch.optim.Optimizer, num_epochs: int | None = None, device: str = 'cpu') -> torch.nn.Module:
    step_num = 0
    epoch_num = 0

    best_checkpoint = None
    best_metric_name = 'dcg@20'
    best_metric_value = float('-inf')


    while epoch_num < num_epochs:
        running_loss = [] # накопление значений лосса

        # итерация по бачам
        for batch in train_dataloader:
            model.train()

            for key in batch:
                if isinstance(batch[key], dict):
                    for sub_key in batch[key]:
                        batch[key][sub_key] = batch[key][sub_key].to(device) # перенос на GPU

                else:
                    assert isinstance(batch[key], torch.Tensor)
                    batch[key] = batch[key].to(device)


            loss = model(batch)['loss']

            loss.backward() # вычисление градиентов
            optimizer.step() # обновление параметров модели
            optimizer.zero_grad() # обнуление градиентов с предыдущих шагов

            step_num += 1

            running_loss.append(loss.item())

            # Запускаем валидацию
            if step_num % 64 == 0:
                pred_valid_indices, true_valid_labels = evaluation(valid_dataloader, model, device)
                valid_metrics = compute_metrics(pred_valid_indices, true_valid_labels)

                if best_metric_value is None or best_metric_value < valid_metrics[best_metric_name]:
                    best_metric_value = valid_metrics[best_metric_name]
                    best_checkpoint = copy.deepcopy(model.state_dict())

                msgs = []
                for metric_name, metrinc_value in valid_metrics.items():
                    msgs.append(f'{metric_name}: {round(metrinc_value, 5)}')
                msg = ', '.join(msgs)
                print(msg)

        print(f'Средний лосс на эпохе #{epoch_num + 1}: {round(np.mean(running_loss), 5)}')

        epoch_num += 1

    print('Обучение завершено!')

    return best_checkpoint

In [69]:
# настройка гиперпараметров
LEARNING_RATE = 0.001
BATCH_SIZE = 256
NUM_EPOCHS = 10

EMBEDDING_DIM = 64
NUM_HEADS = 2
MAX_SEQ_LEN = 200
DROPOUT_RATE = 0.1
NUM_TRANSFORMER_LAYERS = 2

DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'

In [70]:
backbone = SasRecBackbone(
    num_items=NUM_ITEMS,
    embedding_dim=EMBEDDING_DIM,
    num_heads=NUM_HEADS,
    max_seq_len=MAX_SEQ_LEN,
    num_transformer_layers=NUM_TRANSFORMER_LAYERS,
)

In [71]:
# создание датасетов
train_dataset = MovieLensDataset(data=train_data, num_items=NUM_ITEMS, max_seq_len=MAX_SEQ_LEN)
valid_dataset = MovieLensDataset(data=valid_data, num_items=NUM_ITEMS, max_seq_len=MAX_SEQ_LEN)
eval_dataset = MovieLensDataset(data=eval_data, num_items=NUM_ITEMS, max_seq_len=MAX_SEQ_LEN)

# создание даталоадеров
train_dataloader = DataLoader(
    dataset=train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
    collate_fn=collate_fn
)
valid_dataloader = DataLoader(
    dataset=valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    collate_fn=collate_fn
)
eval_dataloader = DataLoader(
    dataset=eval_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    collate_fn=collate_fn
)

model = SasRecReal(backbone=backbone).to(DEVICE) # создание модели и перенос параметров на GPU

# в качестве оптимизатора - Adam
optimizer = torch.optim.Adam(params=model.parameters(), lr=LEARNING_RATE)

# best_checkpoint - словарь с весами, показавшими наилучший результат
best_checkpoint = train(
    train_dataloader=train_dataloader,
    valid_dataloader=valid_dataloader,
    model=model,
    optimizer=optimizer,
    num_epochs=NUM_EPOCHS,
    device=DEVICE
)


model.load_state_dict(best_checkpoint) # загрузка лучших параметров

# оценка на тестовом наборе
pred_eval_indices, true_eval_labels = evaluation(eval_dataloader, model, device=DEVICE)

# вычисление метрик
eval_metrics = compute_metrics(pred_eval_indices, true_eval_labels)

msgs = []
for metric_name, metrinc_value in eval_metrics.items():
    msgs.append(f'{metric_name}: {round(metrinc_value, 5)}')
msg = ', '.join(msgs)
print(msg)

checkpoint_path = './sasrec_full.pth'
torch.save(best_checkpoint, checkpoint_path)


Средний лосс на эпохе #1: 0.64267
Средний лосс на эпохе #2: 0.54449
Средний лосс на эпохе #3: 0.52792
dcg@5: 0.00941, dcg@10: 0.01348, dcg@20: 0.01974, hitrate@5: 0.01552, hitrate@10: 0.02828, hitrate@20: 0.05329, coverage@5: 0.1048, coverage@10: 0.14637, coverage@20: 0.21194
Средний лосс на эпохе #4: 0.51723
Средний лосс на эпохе #5: 0.49156
Средний лосс на эпохе #6: 0.45348
dcg@5: 0.02203, dcg@10: 0.03152, dcg@20: 0.04369, hitrate@5: 0.03691, hitrate@10: 0.06657, hitrate@20: 0.11514, coverage@5: 0.19614, coverage@10: 0.24912, coverage@20: 0.3144
Средний лосс на эпохе #7: 0.42483
Средний лосс на эпохе #8: 0.40257
Средний лосс на эпохе #9: 0.38584
dcg@5: 0.02961, dcg@10: 0.04306, dcg@20: 0.05972, hitrate@5: 0.05037, hitrate@10: 0.09234, hitrate@20: 0.15878, coverage@5: 0.30562, coverage@10: 0.36534, coverage@20: 0.42477
Средний лосс на эпохе #10: 0.37284
Обучение завершено!
dcg@5: 0.02019, dcg@10: 0.0298, dcg@20: 0.04207, hitrate@5: 0.03465, hitrate@10: 0.06457, hitrate@20: 0.11344, co